In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import resample

# 1. Load dataset
file_path = 'TRAINING DATA-SOIL.xlsx'
df = pd.read_excel(file_path, sheet_name=0)

# 2. Encode categorical variables & target
for col in ['Land_use', 'Soil_type']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

target_encoder = LabelEncoder()
df['Overall_class_encoded'] = target_encoder.fit_transform(df['Overall_class'].astype(str))

X = df.drop(columns=['ID', 'Overall_class', 'Overall_class_encoded'])
y = df['Overall_class_encoded']

# 3. Stratified 5-Fold Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

for train_idx, test_idx in skf.split(X, y):
    X_train_raw, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_raw, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Upsample training split only to ensure robust model learning
    train_df = X_train_raw.copy()
    train_df['target'] = y_train_raw
    class_dfs = [train_df[train_df['target'] == i] for i in range(4)]
    expanded_train_dfs = []
    
    target_counts = {0: 300, 1: 300, 2: 300, 3: 300}
    for i, cls_df in enumerate(class_dfs):
        if len(cls_df) > 0:
            resampled_cls = resample(cls_df, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)
        else:
            dummy_row = train_df.iloc[0:1].copy()
            dummy_row['target'] = i
            resampled_cls = resample(dummy_row, replace=True, n_samples=target_counts[i], random_state=42 + i)
            expanded_train_dfs.append(resampled_cls)
            
    train_final = pd.concat(expanded_train_dfs).reset_index(drop=True)
    X_train = train_final.drop(columns=['target'])
    y_train = train_final['target'].values
    
    # Train Random Forest
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Predict probabilities and apply cost-matrix weights to boost minority class recall
    probs = model.predict_proba(X_test)
    # Class order: [Clean, Heavy, Moderate, Slight]
    # Give a slight probability multiplier to minority classes so they trigger correctly
    cost_weights = np.array([2.2, 0.7, 2.2, 2.0])
    calibrated_probs = probs * cost_weights
    y_pred = np.argmax(calibrated_probs, axis=1)
    
    all_y_true.extend(y_test.values)
    all_y_pred.extend(y_pred)

# 4. Aggregate results across all 5 folds for a robust evaluation
acc = accuracy_score(all_y_true, all_y_pred)
print(f"================ Cross-Validated Robust Results ================")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=target_encoder.classes_, zero_division=0))

================ Cross-Validated Robust Results ================
Overall Accuracy: 72.95%

Classification Report:
                        precision    recall  f1-score   support

                 Clean       1.00      0.40      0.57         5
   Heavy contamination       0.92      0.79      0.85        92
Moderate contamination       0.31      0.53      0.39        15
  Slight contamination       0.40      0.60      0.48        10

              accuracy                           0.73       122
             macro avg       0.66      0.58      0.57       122
          weighted avg       0.81      0.73      0.75       122

